# Mall Customer Segmentation - Exploratory Data Analysis & Clustering
This notebook contains the complete EDA, preprocessing, model exploration, and evaluation process for the Mall Customer Segmentation project.

## Project Objectives:
1. Perform Exploratory Data Analysis (EDA) to understand customer demographics and behaviors.
2. Clean and preprocess features (handle missing values, encode Gender, and scale numerical variables).
3. Experiment with clustering algorithms: K-Means, Hierarchical (Agglomerative), and DBSCAN.
4. Utilize the Elbow Method and Silhouette Analysis to identify the optimal number of clusters.
5. Evaluate models using Silhouette Score, Calinski-Harabasz Index, and Davies-Bouldin Index.
6. Export the final K-Means model and scaler for the Streamlit deployment.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Add project root to path for local imports
sys.path.append(os.path.abspath('..'))

from src.utils import load_data, save_plot
from src.preprocessing import preprocess_pipeline
from src.clustering import run_kmeans_search, determine_best_k, train_kmeans, train_hierarchical, train_dbscan, save_model_artifact
from src.evaluate import get_comparison_table

# Set visual settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11


## 1. Load Dataset and Perform Overview Analysis
We load the real Mall Customers dataset and review its shape, basic info, and initial statistical properties.

In [ ]:
# Load dataset (will download automatically from GitHub if not found locally)
df = load_data('../dataset/Mall_Customers.csv')

print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Info:")
df.info()


### Missing Value & Duplicate analysis
We check for missing entries or duplicate records that might skew our model fitting.

In [ ]:
# Missing values
print("Missing values count per column:")
print(df.isnull().sum())

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

# Statistical summary of numerical columns
print("\nStatistical Summary:")
display(df.describe())


## 2. Exploratory Data Analysis (EDA)
We construct visualizations to understand distributions, relationships, and trends between age, income, and spending scores.

In [ ]:
# Age Distribution
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['Age'], kde=True, bins=20, ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Age Distribution of Mall Customers', fontsize=14)
plt.show()

# Gender Distribution
fig, ax = plt.subplots(figsize=(6, 5))
gender_counts = df['Gender'].value_counts()
colors = ['lightcoral', 'cornflowerblue']
ax.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90, colors=colors, explode=(0.03, 0))
ax.set_title('Gender Distribution', fontsize=14)
plt.show()

# Annual Income vs Spending Score Distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df['Annual Income (k$)'], kde=True, bins=15, ax=axes[0], color='salmon', edgecolor='black')
axes[0].set_title('Annual Income Distribution')

sns.histplot(df['Spending Score (1-100)'], kde=True, bins=15, ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_title('Spending Score Distribution')
plt.show()


In [ ]:
# Correlation Heatmap
temp_df = df.copy()
temp_df['Gender_Encoded'] = temp_df['Gender'].map({'Female': 0, 'Male': 1})
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(temp_df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Gender_Encoded']].corr(), 
            annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.show()

# Pairplot colored by Gender
sns.pairplot(df[['Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']], hue='Gender', palette=colors)
plt.show()

# Feature Boxplots by Gender
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(x='Gender', y='Age', data=df, ax=axes[0], palette=colors)
axes[0].set_title('Age by Gender')

sns.boxplot(x='Gender', y='Annual Income (k$)', data=df, ax=axes[1], palette=colors)
axes[1].set_title('Annual Income by Gender')

sns.boxplot(x='Gender', y='Spending Score (1-100)', data=df, ax=axes[2], palette=colors)
axes[2].set_title('Spending Score by Gender')
plt.show()

# Core Spending Behavior Scatter
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Gender', size='Age', 
                sizes=(20, 200), palette=colors, alpha=0.8, data=df, ax=ax)
ax.set_title('Annual Income vs Spending Score (Colored by Gender, Size by Age)', fontsize=14)
plt.show()


## 3. Data Preprocessing & Feature Standardization
To ensure clustering distance metrics are stable and un-biased, we encode categorical inputs and standardize numeric features using `StandardScaler`.

In [ ]:
# Preprocess raw data (handles checks, encodes Gender, standardizes numeric cols)
features_to_scale = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
df_scaled, scaler = preprocess_pipeline(df, features_to_scale=features_to_scale)

print("Preprocessed Data Preview:")
display(df_scaled.head())

# Extract scaled values for modeling
X = df_scaled[features_to_scale].values
print(f"\nClustering feature space shape: {X.shape}")


## 4. Hyperparameter Search & Clustering
We run K-Means across cluster numbers $K \in [2, 10]$ and track the WCSS (Elbow Curve) and average Silhouette Score. We then select the best K, train our final K-Means model, and compare it with Agglomerative Hierarchical and DBSCAN models.

In [ ]:
# Search range for optimal K
k_values, wcss_list, silhouette_scores = run_kmeans_search(X, min_k=2, max_k=10)

# Plots: Elbow Curve vs Silhouette Score
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Curve
axes[0].plot(k_values, wcss_list, marker='o', color='indigo')
axes[0].set_title('K-Means WCSS (Elbow Curve)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_xticks(k_values)

# Silhouette Score
scores_list = [silhouette_scores[k] for k in k_values]
best_k = determine_best_k(silhouette_scores)
axes[1].plot(k_values, scores_list, marker='s', color='teal', linestyle='--')
axes[1].axvline(x=best_k, color='red', linestyle=':', label=f'Optimal K ({best_k})')
axes[1].set_title('Average Silhouette Score')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(k_values)
axes[1].legend()

plt.show()


In [ ]:
# Train final K-Means
kmeans_model, kmeans_labels = train_kmeans(X, best_k)
print(f"K-Means fit successfully with K={best_k} clusters.")

# Train Hierarchical
hier_model, hier_labels = train_hierarchical(X, best_k)
print(f"Agglomerative Hierarchical fit with K={best_k} clusters.")

# Train DBSCAN
dbscan_model, dbscan_labels = train_dbscan(X, eps=0.5, min_samples=5)
print("DBSCAN trained.")


## 5. Model Evaluation and Cluster Visualizations
We compile evaluation metrics for the three algorithms and project our multi-dimensional K-Means clusters onto a 2D space using PCA for plotting.

In [ ]:
# Compare metrics
comparison_df = get_comparison_table(X, kmeans_labels, hier_labels, dbscan_labels)
print("Clustering Metrics comparison:")
display(comparison_df)

# Visualizing K-Means clusters in original coordinates
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=df['Annual Income (k$)'], y=df['Spending Score (1-100)'], 
                hue=kmeans_labels, palette='viridis', style=kmeans_labels, s=100, alpha=0.8, ax=ax)

# Draw cluster centroids
centroids_original = scaler.inverse_transform(kmeans_model.cluster_centers_)
income_idx = features_to_scale.index('Annual Income (k$)')
spend_idx = features_to_scale.index('Spending Score (1-100)')
ax.scatter(centroids_original[:, income_idx], centroids_original[:, spend_idx], 
           s=300, c='red', marker='X', edgecolors='black', label='Centroids')

ax.set_title(f'K-Means Customer Segments (K={best_k})')
ax.set_xlabel('Annual Income (k$)')
ax.set_ylabel('Spending Score (1-100)')
ax.legend()
plt.show()

# 2D PCA projection plot
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans_labels, palette='tab10', s=80, alpha=0.8, ax=ax)
ax.set_title('K-Means Clusters Project on PCA 2D Space')
ax.set_xlabel('PCA Principal Component 1')
ax.set_ylabel('PCA Principal Component 2')
plt.show()


## 6. Save Model Artifacts
We export the final K-Means model and standard scaler instance to the `models/` directory for Streamlit inference.

In [ ]:
# Save artifacts (using project paths)
save_model_artifact(kmeans_model, '../models/kmeans_model.pkl')
save_model_artifact(scaler, '../models/scaler.pkl')
print("Model development pipeline complete!")
